<a href="https://colab.research.google.com/github/marinacornelius/base-para-agentes-de-IA/blob/main/base_para_agentes_de_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base para Agent LangChain

## Setup e Libs

In [2]:
!pip install -qU langchain

In [3]:
!pip install -qU langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 12.6 MB/s eta 0:00:00


In [4]:
!pip install -qU langchain-tavily

In [5]:
!pip install -qU langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.8 MB/s eta 0:00:00


In [6]:
import os
from google.colab import userdata

# 1. Puxa as chaves cadastradas na aba lateral "Secrets" do Colab
google_key = userdata.get('GEMINI_API_KEY')

# 2. Configura as chaves como variáveis de ambiente para o LangChain encontrar
os.environ["GEMINI_API_KEY"] = google_key

print("✅ Chaves carregadas com sucesso a partir dos Secrets do Colab!")

✅ Chaves carregadas com sucesso a partir dos Secrets do Colab!


In [7]:
tavily_api_key = userdata.get('TAVILY_API_KEY')
os.environ["TAVILY_API_KEY"] = tavily_api_key
print("✅ Chaves carregadas com sucesso a partir dos Secrets do Colab!")

✅ Chaves carregadas com sucesso a partir dos Secrets do Colab!


In [8]:
from langchain_tavily import TavilySearch

tavily_web_search_tool = TavilySearch(
  max_results=6,
  #api_key=tavily_api_key
)


In [9]:
query = "qual é o líder do campeonato brasileiro de 2026?"

# Executa a busca
resultados = tavily_web_search_tool.invoke({"query": query})['results']

# Exibe os resultados
for i, res in enumerate(resultados, 1):
    print(f"--- Resultado {i} ---")
    print(f"URL: {res['url']}")
    print(f"Conteúdo: {res['content']}\n")

--- Resultado 1 ---
URL: https://www.instagram.com/reel/DVz1o-zCq4d/
Conteúdo: O São Paulo é o novo líder do Campeonato Brasileiro de 2026, após vitória de 2 a 0 contra a Chapecoense, na estreia do técnico Roger Machado. O

--- Resultado 2 ---
URL: https://www.youtube.com/watch?v=XnSnaN-PLYY
Conteúdo: Terminou nesta quinta-feira a 9ª rodada do Brasileirão. O Palmeiras fez o tema de casa e segue na liderança. O Flamengo não viu a cor da

--- Resultado 3 ---
URL: https://pt.wikipedia.org/wiki/Campeonato_Brasileiro_de_Futebol_de_2026_-_S%C3%A9rie_A
Conteúdo: Campeonato Brasileiro de Futebol de 2026 - Série A ; Palmeiras 5–1 Vitória · Arena Barueri, Barueri 4 de fevereiro, 2.ª rodada. Chapecoense 0–4 Atlético Mineiro

--- Resultado 4 ---
URL: https://www.espn.com.br/futebol/classificacao
Conteúdo: # Campeonato Brasileiro - Classificação 2026. | J | V | E | D | GP | GC | SG | PTS |. ### Glossário. ### Notícias - Série A. Alex Telles admite preocupação com possível 'debandada' no Botafogo e 

# Agente

In [10]:
system_prompt = "Você é um assistente de buscas na web. Você é prestativo e educado. Use as ferramentas disponíveis para responder as demandas do usuário."

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.1)

tools = [tavily_web_search_tool]

In [12]:
from langchain.agents import create_agent

agent = create_agent(model=llm,tools=tools,system_prompt=system_prompt)

In [13]:
agent.invoke({"messages": [("user", "PERGUNTA: quando o homem foi a lua pela primeira e pela última vez?")]})

{'messages': [HumanMessage(content='PERGUNTA: quando o homem foi a lua pela primeira e pela última vez?', additional_kwargs={}, response_metadata={}, id='9890e442-c483-4963-93fb-1222c9da13bc'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "\\u00faltima vez que o homem foi a lua", "search_depth": "basic"}'}, '__gemini_function_call_thought_signatures__': {'82ca716e-db22-412c-bbbf-f24fbf311f53': 'CvcEAQw51sexT5YQzesJRm3CIzjQUO3Hvbz/k8dpdlBSsYdMkKfXRx1yhMYkwQSvD96vbShewPKkMfjZqdtXhLwsGcKPlvVSfju4N9CmeK+AoJL9osiar1JNKaEAIJmOK0nubxh52mtjEDq3//qtdNpdeeDs+FQP4GqfQjiQ4XC0DwYrqz3GJWTfpXwRwRQY3PG1iisx0V75cTiLSyZ5KRhUEUuevRVBDRjnwW/h6L0z6qqFaEC77Nj3pInKij+yIBN3q1WKygvHC7gtcqBnsJV0AIVnVbFwUHTFZi+Ow4NdaEO5MOgsl66nHeg2xCM8LUPrcj2pNmg9otR44U2rqWlICLDDZ74vCdfOP2a//ilMKPqklTyZ+5KUYVmpoy4Hb/EJvDIKuQrbj4F2ibwDLyZww89RbelDw7rrvakIJvAUnh5NML7KOUwXhZmow0XMQrmoLC1J7M5R3bj7J9AxlXoeC+mK5aPPRJ8MP808DAG97g5WV3b0p6unc/IiMffQOc7+y7iiPahvWjQPjPeZ3EBYo

In [14]:
# Usa o .stream() para capturar cada passo do agente em tempo real
print("Iniciando a execução do Agente...\n")
for step in agent.stream({"messages": [("user", "PERGUNTA: Qual a cor predominante dos gatos no Brasil?")]}, stream_mode="updates"):
    # Cada 'step' mostra qual parte do grafo acabou de rodar (pode ser 'agent' ou 'tools')
    for node_name, node_state in step.items():
        print(f"--- Nó executado: {node_name.upper()} ---")

        # Pega a última mensagem gerada neste passo
        ultima_msg = node_state["messages"][-1]

        # Verifica se o modelo decidiu chamar uma ferramenta
        if hasattr(ultima_msg, 'tool_calls') and ultima_msg.tool_calls:
            print(f"🛠️  O agente decidiu chamar a ferramenta: {ultima_msg.tool_calls[0]['name']}")
            print(f"🔎 O que ele digitou na busca: {ultima_msg.tool_calls[0]['args']}")

        # Verifica se é a resposta crua que a ferramenta devolveu
        elif ultima_msg.type == "tool":
            print(f"✅ A ferramenta terminou de pesquisar e devolveu os dados para o agente.")

        print("-" * 40)


# A resposta final será o último print gerado pelo nó 'agent'
print("\n=== RESPOSTA FINAL DO AGENTE ===")
print(ultima_msg.content)

Iniciando a execução do Agente...

--- Nó executado: MODEL ---
🛠️  O agente decidiu chamar a ferramenta: tavily_search
🔎 O que ele digitou na busca: {'query': 'Qual a cor predominante dos gatos no Brasil?'}
----------------------------------------
--- Nó executado: TOOLS ---
✅ A ferramenta terminou de pesquisar e devolveu os dados para o agente.
----------------------------------------
--- Nó executado: MODEL ---
----------------------------------------

=== RESPOSTA FINAL DO AGENTE ===
[{'type': 'text', 'text': 'Não há uma cor de gato *predominante* única e definitiva no Brasil, mas as pesquisas indicam algumas tendências:\n\n*   **Gatos malhadinhos (rajados ou tigrados)** são mencionados como a cor mais popular.\n*   Em domicílios (gatos adotados ou acolhidos), **gatos brancos, cinzas, tricolores e branco-marrons** são mais comuns, o que pode indicar uma preferência por essas cores na hora da adoção.\n*   **Gatos pretos, preto e brancos e laranjas** são encontrados tanto em domicílio